In [ ]:
import ipywidgets as widgets
from IPython.display import display
import threading, tempfile, os, sys, zipfile, requests, shutil, json
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from ipyleaflet import Map, GeoJSON, WidgetControl, LayerGroup

# ─── State ────────────────────────────────────────────────────────────────
state = {
    'temp_dir':       tempfile.mkdtemp(),
    'all_shapefiles': [],
    'shapefile_path': None,
    'predata_path':   None,
    'hbvpara_path':   None,
    'output_files':   {},
    'selected_id':    None,
    'id_col':         None,
    'gdf':            None,
    'geojson_layer':  None,
    'highlight_layer': None,
    'leaflet_map':    None,
}

# ─── Real-time log stream ─────────────────────────────────────────────────
class WidgetStream:
    def __init__(self, out): self.out = out
    def write(self, text): self.out.append_stdout(text)
    def flush(self): pass

# ─── Parse predata.csv ────────────────────────────────────────────────────
def parse_predata(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line == 'input_text':
                continue
            parts = line.split(',', 1)
            if len(parts) == 2:
                rows.append((parts[0].strip(), parts[1].strip()))
    return dict(rows)

# ─── STEP 1: Download shapefile zip ───────────────────────────────────────
shp_url  = widgets.Text(
    placeholder='https://wwwd3.ymparisto.fi/d3/gis_data/spesific/valumaalueet.zip',
    description='URL:',
    layout=widgets.Layout(width='500px')
)
shp_btn  = widgets.Button(description='⬇ Download', button_style='info',
                           layout=widgets.Layout(width='120px'))
shp_prog = widgets.IntProgress(value=0, min=0, max=100,
                                layout=widgets.Layout(width='300px', visibility='hidden'))
shp_lbl  = widgets.Label(value='')

# ─── STEP 2: Shapefile picker ─────────────────────────────────────────────
shp_dropdown   = widgets.Dropdown(description='File:', options=[],
                                   layout=widgets.Layout(width='480px', visibility='hidden'))
shp_load_btn   = widgets.Button(description='🗺 Load Map', button_style='warning',
                                 layout=widgets.Layout(width='120px', visibility='hidden'))
shp_pick_lbl   = widgets.Label(value='')

# ─── STEP 3: Map + search ─────────────────────────────────────────────────
search_box     = widgets.Text(placeholder='Type part of catchment ID to search...',
                               description='Search ID:',
                               layout=widgets.Layout(width='380px', visibility='hidden'))
search_results = widgets.Select(options=[], rows=5,
                                 layout=widgets.Layout(width='380px', visibility='hidden'))
search_lbl     = widgets.Label(value='')
map_container  = widgets.VBox([], layout=widgets.Layout(
    min_height='480px', visibility='hidden',
    border='1px solid #ddd'
))
catchment_lbl  = widgets.Label(value='')

# ─── STEP 4: CSV uploads ──────────────────────────────────────────────────
upload_predata = widgets.FileUpload(accept='.csv', multiple=False,
                                     description='Upload',
                                     layout=widgets.Layout(width='200px'))
upload_hbvpara = widgets.FileUpload(accept='.csv', multiple=False,
                                     description='Upload',
                                     layout=widgets.Layout(width='200px'))
lbl_predata    = widgets.Label(value='No file selected')
lbl_hbvpara    = widgets.Label(value='No file selected')

def _save_csv(upload_w, state_key, label_w, change):
    val = change['new']
    if not val: return
    try:
        uploaded = val[0] if isinstance(val, tuple) else list(val.values())[0]
        fname    = uploaded.get('name', f'{state_key}.csv')
        content  = bytes(uploaded.get('content', b''))
        path     = os.path.join(state['temp_dir'], fname)
        with open(path, 'wb') as f:
            f.write(content)
        state[state_key] = path
        label_w.value = f'✅ {fname}'
    except Exception as e:
        label_w.value = f'❌ {e}'

upload_predata.observe(lambda c: _save_csv(upload_predata, 'predata_path', lbl_predata, c), names='value')
upload_hbvpara.observe(lambda c: _save_csv(upload_hbvpara, 'hbvpara_path', lbl_hbvpara, c), names='value')

# ─── Run / Log / Output ───────────────────────────────────────────────────
project_dir_input = widgets.Text(
    value='/Users/mujaved21/Desktop/WE3Unit/HBV-MODEL',
    description='Project dir:',
    layout=widgets.Layout(width='500px')
)
run_btn  = widgets.Button(description='▶ Run HBV Model', button_style='success',
                           layout=widgets.Layout(width='200px', height='40px'))
run_lbl  = widgets.Label(value='Complete all steps above then click Run.')
log_out  = widgets.Output(layout=widgets.Layout(
    border='1px solid #ccc', height='420px', overflow_y='scroll', padding='10px'
))
file_dd  = widgets.Dropdown(description='Output:', options=[], layout=widgets.Layout(width='420px'))
col_dd   = widgets.Dropdown(description='Column:', options=[], layout=widgets.Layout(width='420px'))
agg_tog  = widgets.ToggleButtons(options=['sum','mean'], description='Aggregate:', value='sum')
plot_btn = widgets.Button(description='📊 Plot on Map', button_style='info')
res_out  = widgets.Output()

# ─── Download handler ─────────────────────────────────────────────────────
def _do_download(b):
    url = shp_url.value.strip()
    if not url:
        shp_lbl.value = '❌ Enter a URL'
        return
    shp_btn.disabled = True
    shp_prog.layout.visibility = 'visible'
    shp_prog.value = 0
    shp_lbl.value  = '⏳ Connecting...'
    def _dl():
        try:
            fname = url.split('?')[0].split('/')[-1] or 'shapefile.zip'
            dest  = os.path.join(state['temp_dir'], fname)
            r = requests.get(url, stream=True, timeout=300)
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            done  = 0
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
                        done += len(chunk)
                        shp_prog.value = int(done/total*100) if total else 50
                        shp_lbl.value  = f'{done/1024/1024:.1f} MB'
            shp_lbl.value += ' — extracting...'
            exdir = os.path.join(state['temp_dir'], 'shp')
            os.makedirs(exdir, exist_ok=True)
            with zipfile.ZipFile(dest, 'r') as z:
                z.extractall(exdir)
            shps = [os.path.join(dp, fn)
                    for dp, _, fns in os.walk(exdir)
                    for fn in fns if fn.endswith('.shp')]
            if not shps:
                shp_lbl.value = '❌ No .shp found in zip'
                return
            state['all_shapefiles'] = shps
            shp_dropdown.options = {os.path.basename(s): s for s in shps}
            shp_dropdown.layout.visibility  = 'visible'
            shp_load_btn.layout.visibility  = 'visible'
            shp_prog.value = 100
            shp_lbl.value  = f'✅ Found {len(shps)} shapefiles — pick one below'
        except Exception as e:
            shp_lbl.value = f'❌ {e}'
        finally:
            shp_btn.disabled = False
    threading.Thread(target=_dl).start()

shp_btn.on_click(_do_download)

# ─── Highlight a catchment on the leaflet map ─────────────────────────────
def _highlight(selected_id):
    gdf    = state['gdf']
    id_col = state['id_col']
    m      = state['leaflet_map']
    if m is None or gdf is None:
        return
    # Remove old highlight
    if state['highlight_layer']:
        m.remove_layer(state['highlight_layer'])
    # Build highlight GeoJSON
    sel = gdf[gdf[id_col] == selected_id].to_crs('EPSG:4326')[[id_col, 'geometry']].copy()
    sel[id_col] = sel[id_col].astype(str)
    if sel.empty:
        return
    hl = GeoJSON(
        data=json.loads(sel.to_json()),
        style={'color': 'orange', 'fillColor': 'orange',
               'fillOpacity': 0.6, 'weight': 3}
    )
    m.add_layer(hl)
    state['highlight_layer'] = hl
    # Zoom to it
    bounds = sel.total_bounds  # minx miny maxx maxy
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    state['selected_id']  = selected_id
    catchment_lbl.value   = f'✅ Selected catchment ID: {selected_id} — ready to run'

# ─── Load shapefile + build leaflet map ───────────────────────────────────
def _load_shapefile(b):
    path = shp_dropdown.value
    if not path:
        shp_pick_lbl.value = '❌ Select a shapefile first'
        return
    shp_load_btn.disabled = True
    shp_pick_lbl.value = '⏳ Loading shapefile...'
    threading.Thread(target=_load_shapefile_thread, args=(path,)).start()

def _load_shapefile_thread(path):
    try:
        gdf = gpd.read_file(path)
        id_col = next((c for c in gdf.columns if c.lower().endswith('_id')), None)
        if not id_col:
            shp_pick_lbl.value = '❌ No *_id column in this shapefile'
            return
        state['shapefile_path'] = path
        state['gdf']     = gdf
        state['id_col']  = id_col

        # Reproject to WGS84 for leaflet
        gdf_wgs = gdf.to_crs('EPSG:4326')
        bounds  = gdf_wgs.total_bounds
        center  = [(bounds[1]+bounds[3])/2, (bounds[0]+bounds[2])/2]

        # Strip to only id + geometry
        gdf_wgs = gdf_wgs[[id_col, 'geometry']].copy()
        gdf_wgs[id_col] = gdf_wgs[id_col].astype(str)

        # Simplify geometry based on number of catchments — keeps browser fast
        n = len(gdf_wgs)
        if n > 2000:
            tol = 0.005
        elif n > 500:
            tol = 0.002
        elif n > 100:
            tol = 0.001
        else:
            tol = 0.0
        if tol > 0:
            gdf_wgs['geometry'] = gdf_wgs['geometry'].simplify(tol, preserve_topology=True)
            shp_pick_lbl.value = f'⏳ Simplified {n} catchments (tol={tol}) — building map...'

        # Build map
        m = Map(center=center, zoom=7,
                layout=widgets.Layout(width='100%', height='480px'))

        # Add all catchments as GeoJSON
        geo_layer = GeoJSON(
            data=json.loads(gdf_wgs.to_json()),
            style={'color': 'black', 'fillColor': '#a8d5f5',
                   'fillOpacity': 0.4, 'weight': 1},
            hover_style={'fillColor': '#f5c842', 'fillOpacity': 0.7}
        )

        # Click handler
        def on_click(event=None, feature=None, **kwargs):
            if feature is None:
                return
            props = feature.get('properties', {})
            clicked_id = props.get(id_col)
            if clicked_id is not None:
                _highlight(clicked_id)

        geo_layer.on_click(on_click)
        m.add_layer(geo_layer)
        state['geojson_layer'] = geo_layer
        state['leaflet_map']   = m
        state['highlight_layer'] = None

        # Show map by injecting directly into VBox
        map_container.children = [m]
        map_container.layout.visibility = 'visible'

        # Enable search
        all_ids = [str(v) for v in gdf[id_col].unique()]
        state['all_ids'] = all_ids
        search_box.layout.visibility     = 'visible'
        search_results.layout.visibility = 'visible'
        search_results.options = all_ids[:50]
        search_lbl.value = f'Showing first 50 of {len(all_ids)} — type to filter, click to select'

        shp_pick_lbl.value = (f'✅ {os.path.basename(path)} loaded | '
                               f'ID column: {id_col} | {len(gdf)} catchments | '
                               f'Click a catchment OR search below')
        catchment_lbl.value = '👆 Click a catchment on the map, or search by ID below'

    except Exception as e:
        import traceback
        shp_pick_lbl.value = f'❌ {e}'
        print(traceback.format_exc())
    finally:
        shp_load_btn.disabled = False

shp_load_btn.on_click(_load_shapefile)

# ─── Search box logic ─────────────────────────────────────────────────────
def _on_search(change):
    q = change['new'].strip()
    all_ids = state.get('all_ids', [])
    if not q:
        search_results.options = all_ids[:50]
        search_lbl.value = f'Showing first 50 of {len(all_ids)}'
        return
    matches = [i for i in all_ids if q in i]
    search_results.options = matches[:100]
    search_lbl.value = f'{len(matches)} match(es) — click one to select'

search_box.observe(_on_search, names='value')

# ─── Clicking a search result selects + highlights it ─────────────────────
def _on_select(change):
    val = change['new']
    if val is None or val == '':
        return
    gdf    = state.get('gdf')
    id_col = state.get('id_col')
    if gdf is None or id_col is None:
        return
    # Try string match first, then float
    match = gdf[gdf[id_col].astype(str) == str(val)]
    if match.empty:
        try:
            match = gdf[gdf[id_col] == float(val)]
        except (ValueError, TypeError):
            pass
    if not match.empty:
        selected = match[id_col].values[0]
        _highlight(selected)
        search_lbl.value = f'✅ Selected: {selected}'
    else:
        search_lbl.value = f'❌ ID {val} not found in shapefile'

search_results.observe(_on_select, names='value')

# ─── Build input tab ──────────────────────────────────────────────────────
input_tab = widgets.VBox([
    widgets.HTML('<h3>📂 Step 1 — Download Shapefile</h3>'),
    widgets.HBox([shp_url, shp_btn]),
    widgets.HBox([shp_prog, shp_lbl]),

    widgets.HTML('<h3>🗂 Step 2 — Pick a Shapefile</h3>'),
    widgets.HBox([shp_dropdown, shp_load_btn]),
    shp_pick_lbl,

    widgets.HTML('<h3>📍 Step 3 — Select Catchment</h3>'),
    map_container,
    catchment_lbl,
    widgets.HTML('<b>Search by ID:</b>'),
    search_box,
    search_lbl,
    search_results,

    widgets.HTML('<h3>📄 Step 4 — Upload CSV Files</h3>'),
    widgets.HTML('<b>predata.csv</b> — paths to NetCDF + land-use files'),
    widgets.HBox([upload_predata, lbl_predata]),
    widgets.HTML('<b>hbv_para.csv</b> — model parameters'),
    widgets.HBox([upload_hbvpara, lbl_hbvpara]),

    widgets.HTML('<hr>'),
    widgets.HTML('<b>Output folder</b> — where to save results'),
    project_dir_input,
    widgets.HTML('<hr>'),
    run_btn,
    run_lbl,
], layout=widgets.Layout(padding='16px'))

log_tab = widgets.VBox([
    widgets.HTML('<h3>📋 Model Logs</h3>'),
    log_out
], layout=widgets.Layout(padding='16px'))

output_tab = widgets.VBox([
    widgets.HTML('<h3>📈 Results</h3>'),
    file_dd, col_dd, agg_tog, plot_btn, res_out
], layout=widgets.Layout(padding='16px'))

tabs = widgets.Tab(children=[input_tab, log_tab, output_tab])
tabs.set_title(0, '📂 Input')
tabs.set_title(1, '📋 Logs')
tabs.set_title(2, '📈 Output')
display(tabs)

# ─── Run model ────────────────────────────────────────────────────────────
def run_model(b):
    if not state.get('selected_id'):
        run_lbl.value = '❌ Please select a catchment first (Step 3)'
        return
    run_btn.disabled = True
    run_lbl.value    = '⏳ Running...'
    log_out.clear_output()
    res_out.clear_output()
    tabs.selected_index = 1

    def _run():
        orig = sys.stdout
        sys.stdout = WidgetStream(log_out)
        try:
            if not state['shapefile_path']:
                print('❌ No shapefile loaded'); return
            if not state['predata_path']:
                print('❌ Please upload predata.csv'); return
            if not state['hbvpara_path']:
                print('❌ Please upload hbv_para.csv'); return

            print(f'✅ Shapefile:    {os.path.basename(state["shapefile_path"])}')
            print(f'✅ Catchment ID: {state["selected_id"]}')
            print(f'✅ predata.csv:  {os.path.basename(state["predata_path"])}')
            print(f'✅ hbv_para.csv: {os.path.basename(state["hbvpara_path"])}')

            print('\n🔧 Parsing predata.csv...')
            cfg = parse_predata(state['predata_path'])
            print('   Keys:', list(cfg.keys()))

            for key, fname in [('output_csv_path','met_input.csv'),
                                ('csv_parameters_path','hbv_para.csv')]:
                if key in cfg and not os.path.isabs(cfg[key]):
                    cfg[key] = os.path.join(state['temp_dir'], fname)
                    print(f'   {key} → {cfg[key]}')

            src = os.path.abspath(state['hbvpara_path'])
            dst = os.path.abspath(cfg['csv_parameters_path'])
            if src != dst:
                shutil.copy(src, dst)
                print(f'   Copied hbv_para.csv → {dst}')
            else:
                print('   hbv_para.csv already in place')

            print('\n⚙️  Running hbv_prepare...')
            from hbv_prepare import prepare_meteorological_and_landuse_data_direct
            output_csv = prepare_meteorological_and_landuse_data_direct(
                shapefile_path         = state['shapefile_path'],
                catchment_id_name      = state['id_col'],
                taso_id_of_interest    = state['selected_id'],
                precipitation_nc       = cfg['precipitation_nc_file_path'],
                evapotranspiration_nc  = cfg['evapotranspiration_nc_file_path'],
                temperature_nc         = cfg['temperature_nc_file_path'],
                output_csv_path        = cfg['output_csv_path'],
                urban_land_path        = cfg['urban_land_path'],
                agricultural_land_path = cfg['agricultural_land_path'],
                csv_parameters_path    = cfg['csv_parameters_path'],
            )
            print(f'✅ Met data: {output_csv}')

            print('\n🌊 Running HBV model...')
            # Add project dir to path so hbv_S2S.py can be found
            project_dir = os.path.dirname(os.path.abspath(state['hbvpara_path']))
            if project_dir not in sys.path:
                sys.path.insert(0, project_dir)
            # Also add notebook dir (where hbv_S2S.py likely lives)
            notebook_dir = os.getcwd()
            if notebook_dir not in sys.path:
                sys.path.insert(0, notebook_dir)
            os.chdir(os.path.dirname(cfg['csv_parameters_path']))
            from hbv_S2S import run_hbv_model
            output_files = run_hbv_model(output_csv)
            state['output_files'] = output_files
            print('\n✅ HBV model complete!')
            for k, v in output_files.items():
                print(f'   {k}: {v}')

            file_dd.options     = {k: v for k, v in output_files.items()}
            tabs.selected_index = 2
            run_lbl.value       = '✅ Done! Check the Output tab.'

            # ── Copy outputs to project folder ──
            project_dir = project_dir_input.value.strip()
            os.makedirs(project_dir, exist_ok=True)
            print(f'\n📁 Copying output files to: {project_dir}')
            for k, src in output_files.items():
                dst = os.path.join(project_dir, os.path.basename(src))
                if os.path.abspath(src) != os.path.abspath(dst):
                    shutil.copy(src, dst)
                print(f'   ✅ {os.path.basename(src)}')
            met_dst = os.path.join(project_dir, 'met_input.csv')
            if os.path.abspath(cfg['output_csv_path']) != os.path.abspath(met_dst):
                shutil.copy(cfg['output_csv_path'], met_dst)
            print(f'   ✅ met_input.csv')
            print(f'\n📂 All outputs saved to: {project_dir}')

        except Exception as e:
            import traceback
            print(f'\n❌ Error: {e}')
            print(traceback.format_exc())
            run_lbl.value = f'❌ {e}'
        finally:
            sys.stdout       = orig
            run_btn.disabled = False

    threading.Thread(target=_run).start()

run_btn.on_click(run_model)

# ─── Output tab ───────────────────────────────────────────────────────────
def on_file_change(change):
    if change['new']:
        df = pd.read_csv(change['new'])
        col_dd.options = df.select_dtypes(include='number').columns.tolist()

file_dd.observe(on_file_change, names='value')

def on_plot(b):
    res_out.clear_output()
    with res_out:
        try:
            df  = pd.read_csv(file_dd.value)
            col = col_dd.value
            agg = agg_tog.value
            val = df[col].mean() if agg == 'mean' else df[col].sum()
            print(f'{agg.title()} of "{col}": {val:.4f}')
            gdf = gpd.read_file(state['shapefile_path'])
            gdf['plot_value'] = 0.0
            gdf.loc[gdf[state['id_col']] == state['selected_id'], 'plot_value'] = val
            fig, ax = plt.subplots(figsize=(8, 6))
            gdf.plot(ax=ax, column='plot_value', cmap='Blues',
                     legend=True, edgecolor='black',
                     missing_kwds={'color': 'lightgrey'})
            ax.set_title(f'{agg.title()} of {col} — Catchment {state["selected_id"]}')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'❌ Plot error: {e}')

plot_btn.on_click(on_plot)